This notebook is for stuying how Resilient Distributed Dataset (RDD) works behind the scene in Spark.

# Notebook setup

In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession

# Fix for "Python worker failed to connect back" on Windows/local environments
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [3]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [4]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

# Filter
This is how `WHERE` works in spark SQL.

In [5]:
from datetime import datetime

In [6]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [ ]:
rows = rdd.filter(filter_outliers).take(10)
rows[0] ## first row after filtered

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 3, 19, 0, 1), PULocationID=244, total_amount=8.8)

In [11]:
rows

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 3, 19, 0, 1), PULocationID=244, total_amount=8.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 19, 55, 1), PULocationID=166, total_amount=8.16),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 10, 20, 42), PULocationID=145, total_amount=3.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 23, 10, 24), PULocationID=101, total_amount=31.79),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 12, 17, 30, 50), PULocationID=74, total_amount=5.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 27, 19, 33, 9), PULocationID=134, total_amount=12.09),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 24, 21, 25, 12), PULocationID=42, total_amount=7.25),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 17, 6, 40), PULocationID=82, total_amount=19.56),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 7, 5, 46), PULocationID=222, total_amount=29.88),
 Row(lpep_pickup_datetime=datetime.datetime(2020

# Group By
`GROUP BY` and aggregations have several underlying processes. In Spark, the first stage is mapping and reducing key in each partition (local reduce). Then, same key will be shuffle to the same partition. Finally, the aggregation will be perform for each key. Below are functions that works.
- `map()`: Prepares data into `(key, value)` pairs. To calculate averages or counts, the value is often a tuple itself, like `(amount, 1)`.
- `reduceByKey()`: Performs the calculation. It is associative, meaning `(a + b) + c` must equal `a + (b + c)`.
- Final Formatting: Use `.map()` to move the keys and aggregated values back into a flat structure (e.g., `(hour, zone, total_revenue)`).

In [ ]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [14]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [13]:
from collections import namedtuple

In [15]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [16]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [18]:
from pyspark.sql import types
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [19]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema) 

In [20]:
df_result.write.parquet('tmp/green-revenue')

# Map Partition
`mapPartition()` allows us to process an entire partition of data as a single iterator, which is particularly useful for expensive initializations like loading a Machine Learning (ML) model once per partition rather than once per row.

In [21]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [22]:
import pandas as pd

In [23]:
rows = duration_rdd.take(10)

In [24]:
df = pd.DataFrame(rows, columns=columns)

In [25]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [26]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [27]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [28]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [29]:
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
|               5.0|
|               4.1|
|               0.0|
|              24.4|
|1.7000000000000002|
|              7.95|
|               3.7|
|              21.7|
|28.700000000000003|
|              32.5|
|               4.9|
|             13.25|
|              13.4|
|               0.0|
|             52.75|
|             26.25|
|             31.75|
|              17.5|
|              10.2|
|              14.0|
+------------------+
only showing top 20 rows
